In [1]:
import heapq
from collections import defaultdict

def anomaly_scores(arr, k):
    low = []
    high = []
    delayed = defaultdict(int)

    def prune(heap):
        while heap:
            num = -heap[0] if heap is low else heap[0]
            if delayed[num] > 0:
                delayed[num] -= 1
                heapq.heappop(heap)
            else:
                break

    def rebalance():
        if len(low) > len(high) + 1:
            heapq.heappush(high, -heapq.heappop(low))
        elif len(low) < len(high):
            heapq.heappush(low, -heapq.heappop(high))

    def get_median():
        if k % 2 == 1:
            return -low[0]
        else:
            return (-low[0] + high[0]) /2
        
    result = []

    for i in range(len(arr)):
        if not low or arr[i] <= -low[0]:
            heapq.heappush(low, -arr[i])
        else:
            heapq.heappush(high, arr[i])

        rebalance()

        if i >= k:
            out = arr[i - k]
            delayed[out] += 1

            if out <= -low[0]:
                prune(low)
            else:
                prune(high)

            rebalance()

        if i >=k:
            median = get_median()
            result.append(abs(arr[i] - median))

    return result
            
def main():
    arr = [10, 20, 30, 40, 100]
    k = 3

    scores = anomaly_scores(arr, k)

    print("Array: ", arr)
    print("Window size: ", k)
    print("Anomaly Scores: ", scores)

if __name__ == "__main__":
    main()
                            

Array:  [10, 20, 30, 40, 100]
Window size:  3
Anomaly Scores:  [20, 70]


In [3]:
import heapq
import math
def top_k_cor(data, k):
    n = len(data)
    m = len(data[0])

    features = list(zip(*data))

    means = []
    stds = []

    for f in features:
        mean = sum(f)/n
        variance = sum((x - mean) ** 2 for x in f) /n
        std = math.sqrt(variance)
        means.append(mean)
        stds.append(std)

    min_heap = []
    for i in range(m):
        for j in range(i + 1, m):
            if stds[i] == 0 or stds[j] == 0:
                continue  

            cov = sum(
                (features[i][r] - means[i]) * (features[j][r] - means[j])
                for r in range(n)
            ) / n

            corr = cov / (stds[i] * stds[j])
            abs_corr = abs(corr)

            heapq.heappush(min_heap, (abs_corr, i, j, corr))

            if len(min_heap) > k:
                heapq.heappop(min_heap)

    result = []
    while min_heap:
        abs_corr, i, j, corr = heapq.heappop(min_heap)
        result.append((i, j, round(corr, 3)))

    result.reverse()  
    return result